# Queue workers
Workers take jobs as capacity becomes available; they do not wait for a whole batch.


In [ ]:
# Each free worker immediately takes the next job; there is no batch barrier.
import asyncio

queue: asyncio.Queue[int | None] = asyncio.Queue()

async def worker(name: str) -> None:
    while (job := await queue.get()) is not None:
        await asyncio.sleep(0.01)
        print(name, job)
        # Acknowledge completion so queue.join() knows when all work is done.
        queue.task_done()

workers = [asyncio.create_task(worker(f"w{i}")) for i in range(3)]
for job in range(6):
    await queue.put(job)
await queue.join()
for _ in workers:
    await queue.put(None)
await asyncio.gather(*workers)


## Polished version
A worker pool gives bounded concurrency and shuts down cleanly with sentinels.


In [ ]:
# WorkerPool owns bounded concurrency, queue lifecycle, and clean shutdown.
from collections.abc import Awaitable, Callable
from dataclasses import dataclass

@dataclass(frozen=True)
class Job:
    id: int
    prompt: str

class WorkerPool:
    def __init__(self, concurrency: int, handler: Callable[[Job], Awaitable[None]]) -> None:
        if concurrency < 1:
            raise ValueError("concurrency must be positive")
        self.concurrency = concurrency
        self.handler = handler
        self.queue: asyncio.Queue[Job | None] = asyncio.Queue()

    async def _worker(self) -> None:
        while True:
            job = await self.queue.get()
            try:
                if job is None:
                    return
                await self.handler(job)
            finally:
                self.queue.task_done()

    async def run(self, jobs: list[Job]) -> None:
        # The fixed task count is the maximum number of simultaneous jobs.
        tasks = [asyncio.create_task(self._worker()) for _ in range(self.concurrency)]
        for job in jobs:
            await self.queue.put(job)
        await self.queue.join()
        # One sentinel stops each worker after all real jobs finish.
        for _ in tasks:
            await self.queue.put(None)
        await asyncio.gather(*tasks)

async def handle(job: Job) -> None:
    await asyncio.sleep(0.01)
    print(f"completed {job.id}")

jobs = [Job(i, f"prompt-{i}") for i in range(6)]
await WorkerPool(concurrency=3, handler=handle).run(jobs)
